In [5]:
import sys
print(sys.executable)


c:\Users\isabe\anaconda3\python.exe


In [2]:
!pip install psycopg2-binary


The %sql command uses SQLAlchemy to connect the database, but it needs a separate driver to communicate with PostgreSQL, which is psycopg2. 

In [ ]:
!pip install jupysql

In [2]:
%load_ext sql
%sql postgresql://postgres@localhost:5432/northwind


Connecting to 'postgresql://postgres@localhost:5432/northwind'

In [12]:
%sql SELECT * FROM customers LIMIT 5;


Running query in 'postgresql://postgres@localhost:5432/northwind'

5 rows affected.

customer_id,company_name,contact_name,contact_title,address,city,region,postal_code,country,phone,fax
ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,None,12209,Germany,030-0074321,030-0076545
ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,None,05021,Mexico,(5) 555-4729,(5) 555-3745
ANTON,Antonio Moreno Taquería,Antonio Moreno,Owner,Mataderos 2312,México D.F.,None,05023,Mexico,(5) 555-3932,None
AROUT,Around the Horn,Thomas Hardy,Sales Representative,120 Hanover Sq.,London,None,WA1 1DP,UK,(171) 555-7788,(171) 555-6750
BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Berguvsvägen 8,Luleå,None,S-958 22,Sweden,0921-12 34 65,0921-12 34 67


In [13]:
%%sql
SELECT table_name AS name,
       table_type AS type
  FROM information_schema.tables
 WHERE table_schema = 'public' AND table_type IN ('BASE TABLE', 'VIEW');

Running query in 'postgresql://postgres@localhost:5432/northwind'

14 rows affected.

name,type
territories,BASE TABLE
order_details,BASE TABLE
employee_territories,BASE TABLE
us_states,BASE TABLE
customers,BASE TABLE
orders,BASE TABLE
employees,BASE TABLE
shippers,BASE TABLE
products,BASE TABLE
categories,BASE TABLE


In [15]:
%%sql
ALTER TABLE employees
DROP COLUMN photo;

Running query in 'postgresql://postgres@localhost:5432/northwind'

++
||
++
++

In [ ]:
# %%sql
# DROP VIEW IF EXISTS detailed_order;

Running query in 'postgresql://postgres@localhost:5432/northwind'

++
||
++
++

### Combine Customers and Orders

I joined the `customers` and `orders` tables on `customer_id` and saved the result as the `detailed_order` view. This combines customer details (company, contact, country, city, region) with order details (order date, shipped date, ship destination, employee) into a single view, so future analysis won't require repeating this join.

In [7]:
%%sql
CREATE VIEW detailed_order AS
       SELECT c.customer_id,
       c.country,
       c.city,
       c.region,
       c.company_name,
       c.contact_name,
       o.order_id,
       o.order_date,
       o.shipped_date,
       o.ship_country,
       o.ship_city,
       o.employee_id
         FROM customers as c
         JOIN orders as o
           ON c.customer_id=o.customer_id;
SELECT * FROM detailed_order;


Running query in 'postgresql://postgres@localhost:5432/northwind'

830 rows affected.

customer_id,country,city,region,company_name,contact_name,order_id,order_date,shipped_date,ship_country,ship_city,employee_id
VINET,France,Reims,None,Vins et alcools Chevalier,Paul Henriot,10248,1996-07-04,1996-07-16,France,Reims,5
TOMSP,Germany,Münster,None,Toms Spezialitäten,Karin Josephs,10249,1996-07-05,1996-07-10,Germany,Münster,6
HANAR,Brazil,Rio de Janeiro,RJ,Hanari Carnes,Mario Pontes,10250,1996-07-08,1996-07-12,Brazil,Rio de Janeiro,4
VICTE,France,Lyon,None,Victuailles en stock,Mary Saveley,10251,1996-07-08,1996-07-15,France,Lyon,3
SUPRD,Belgium,Charleroi,None,Suprêmes délices,Pascale Cartrain,10252,1996-07-09,1996-07-11,Belgium,Charleroi,4
HANAR,Brazil,Rio de Janeiro,RJ,Hanari Carnes,Mario Pontes,10253,1996-07-10,1996-07-16,Brazil,Rio de Janeiro,3
CHOPS,Switzerland,Bern,None,Chop-suey Chinese,Yang Wang,10254,1996-07-11,1996-07-23,Switzerland,Bern,5
RICSU,Switzerland,Genève,None,Richter Supermarkt,Michael Holz,10255,1996-07-12,1996-07-15,Switzerland,Genève,9
WELLI,Brazil,Resende,SP,Wellington Importadora,Paula Parente,10256,1996-07-15,1996-07-17,Brazil,Resende,3
HILAA,Venezuela,San Cristóbal,Táchira,HILARION-Abastos,Carlos Hernández,10257,1996-07-16,1996-07-22,Venezuela,San Cristóbal,4


### Add Product Details to Orders

I joined `order_details`, `products`, and `orders` (via the `detailed_order` view) to bring in the product name and quantity for each order line, saving the result as the `detailed_order_info` view. This gives a complete, line-item-level view of what was ordered, so future analysis won't require repeating this join.

In [9]:
%%sql
CREATE VIEW detailed_order_info as 
      SELECT od.order_id, 
             od.product_id,
             od.quantity,
             p.product_name
        FROM detailed_order as de
        JOIN employees as e
          ON e.employee_id=de.employee_id
        JOIN order_details as od
          ON de.order_id=od.order_id
        JOIN products as p
          ON p.product_id=od.product_id;
       
SELECT * FROM detailed_order_info

Running query in 'postgresql://postgres@localhost:5432/northwind'

2155 rows affected.

order_id,product_id,quantity,product_name
10248,11,12,Queso Cabrales
10248,42,10,Singaporean Hokkien Fried Mee
10248,72,5,Mozzarella di Giovanni
10249,14,9,Tofu
10249,51,40,Manjimup Dried Apples
10250,41,10,Jack's New England Clam Chowder
10250,51,35,Manjimup Dried Apples
10250,65,15,Louisiana Fiery Hot Pepper Sauce
10251,22,6,Gustaf's Knäckebröd
10251,57,15,Ravioli Angelo


### Combine Orders with Employees

I joined `orders` with `customers` and `employees`, saving the result as the `orders_with_employees` view. I used a `LEFT JOIN` on `employees` (instead of an inner join) so that every order is kept even if its `employee_id` doesn't match a valid employee — for example due to a system error or missing data. This way, orders with no matching employee still show up with `NULL` employee fields instead of being silently dropped, making it easier to spot and investigate that kind of data issue.

In [10]:
%%sql
CREATE VIEW orders_with_employees AS
SELECT 
    o.order_id,
    c.customer_id, 
    c.contact_name, 
    e.employee_id,
    e.first_name || ' ' || e.last_name AS employee_full_name,
    e.title

FROM orders AS o  
JOIN customers AS c
    ON o.customer_id = c.customer_id
LEFT JOIN employees AS e
    ON o.employee_id = e.employee_id;

SELECT * from orders_with_employees

Running query in 'postgresql://postgres@localhost:5432/northwind'

830 rows affected.

order_id,customer_id,contact_name,employee_id,employee_full_name,title
10248,VINET,Paul Henriot,5,Steven Buchanan,Sales Manager
10249,TOMSP,Karin Josephs,6,Michael Suyama,Sales Representative
10250,HANAR,Mario Pontes,4,Margaret Peacock,Sales Representative
10251,VICTE,Mary Saveley,3,Janet Leverling,Sales Representative
10252,SUPRD,Pascale Cartrain,4,Margaret Peacock,Sales Representative
10253,HANAR,Mario Pontes,3,Janet Leverling,Sales Representative
10254,CHOPS,Yang Wang,5,Steven Buchanan,Sales Manager
10255,RICSU,Michael Holz,9,Anne Dodsworth,Sales Representative
10256,WELLI,Paula Parente,3,Janet Leverling,Sales Representative
10257,HILAA,Carlos Hernández,4,Margaret Peacock,Sales Representative
